<a href="https://colab.research.google.com/github/ankitsingh435517/nn/blob/main/makemore2_impl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Spell out the whole training process, eval process and sampling process
# Then write the pseudo code for the exact thing
# Then write the reasoning for all the pseudo code
# Then replace it with actual code - seek docs help
# Verify for the model to have been covered all the parts (training, eval, sampling) - verify with the video

In [ ]:
# Spell out the whole training process, eval process and sampling process
#
# Goal: We need to train a neural network which can see 3 chars as in the context length is 3 chars and can output a single char which is high probable based on the training dataset statistical relationship.
# we need to build out a neural network with 2 layers 1st will be the input layer as in it will process inputs directly also called the hidden layer
# denoted by h
# next layer will be the output layer which will take the neurons of previous layer as input and will process all the neurons and will generate 27 chars as its output as we want a single char from a to z
# we then apply softmax to get the normalized char distribution which can then be sampled from the neural network.
# we also need biases for each layer for each neuron so
# let's say we have a single input called '...' we need a way to feed this input of chars into something nn understands and can transform it.
# so we transform it into integers and those integers can be further transformed into 2 dimensional vectors (here dimensions can change depending on how the training goes if the data is more complex for the nn to represent) - its one of the knobs of nn to tweak
# so we now have 2 dimensional vector for each char and a single input fed will have 2 * 3 = 6 vector inputs as in x1 to x6.
# now we have 6 inputs so we would need 6 weights and a single bias to interact with and that makes up a single neuron and we can have 100 such neurons to start with. (Again the number of neurons can be increased later as the training demands) - another knob of the nn.
# then we would need a lookup table of character embeddings of 2 dimensions from a to z to get the embeddings for a particular char which will be used in the computation - the weights and biases will be transformed with respect to the inputs given and based on that the lookup table will be transformed as in the chars will be grouped likewise
# We then need to train it in batches, since the data is statistical it still makes sense to pick a random batch and do computations on that and then update the weights, biases and lookup table and repeat it for some turns say for 10k steps
# then do the eval for eval we need our dataset to be compiled into 3 types 80% to be train split, 10% dev split to eval, 10% test split
# now we need to create a matrix of say 32 input features of 3 chars of 2 dimensions each embedding to start with and we store the next char in Ytr
# similarly we put 10% of the same thing after Xtr in Xdev and lastly in Xtest and respective Y
# we then do the eval on whole Xtr then eval on Xdev then test only few times to avoid the model overfit the test split too.
# for eval we run the forward pass and find the loss and do not perform the backprop, the loss here is cross_entropy as it is more efficient and stable numerically
# then lastly we sample from our model once the loss is low enough if not low enough train some more time and tweak the knobs based on intuitive debugging and plotting the embeddings learnt in the lookup table.
# for sampling we give a starting char say '.' its embedding is fed into the neural net and we take a multinomial distrbution and create a word we create few words and examine and see if it makes sense.
# in all the random generation or sampling processes use a same manual seed for generators.

In [ ]:
# pseudocode (step by step)
#
#
#
# Step 1: data loading and parsing
# 1.1 Fetch the data from the source
import requests

url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
response = requests.get(url)
words = response.text.splitlines()
words

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn',
 'abigail',
 'emily',
 'elizabeth',
 'mila',
 'ella',
 'avery',
 'sofia',
 'camila',
 'aria',
 'scarlett',
 'victoria',
 'madison',
 'luna',
 'grace',
 'chloe',
 'penelope',
 'layla',
 'riley',
 'zoey',
 'nora',
 'lily',
 'eleanor',
 'hannah',
 'lillian',
 'addison',
 'aubrey',
 'ellie',
 'stella',
 'natalie',
 'zoe',
 'leah',
 'hazel',
 'violet',
 'aurora',
 'savannah',
 'audrey',
 'brooklyn',
 'bella',
 'claire',
 'skylar',
 'lucy',
 'paisley',
 'everly',
 'anna',
 'caroline',
 'nova',
 'genesis',
 'emilia',
 'kennedy',
 'samantha',
 'maya',
 'willow',
 'kinsley',
 'naomi',
 'aaliyah',
 'elena',
 'sarah',
 'ariana',
 'allison',
 'gabriella',
 'alice',
 'madelyn',
 'cora',
 'ruby',
 'eva',
 'serenity',
 'autumn',
 'adeline',
 'hailey',
 'gianna',
 'valentina',
 'isla',
 'eliana',
 'quinn',
 'nevaeh',
 'ivy',
 'sadie',
 'piper',
 'lydia',
 'alexa',
 'josephine',
 'emery',
 'julia'

In [ ]:
# 1.2 parse it into string to int and int to str
chars = sorted(set(list('.' + ''.join(words))))
stoi, itos = {}, {}
for i,s in enumerate(chars):
  stoi[s] = i
  itos[i] = s

In [119]:
# Step 2: Compile dataset into training, dev and train split (inputs & outputs)
# shuffle the dataset randomly to avoid gradient to form premature bias over inherent structures and avoid the gradient oscillations, said that if the data lacks structure already then not needed.
# 2.1 split 80% train, 10% dev, 10% test
import random
import torch

random.seed(42) # predictable manual seed

# shuffle via sampling without replacement
shuffled_words = random.sample(words, len(words))

l1 = int(0.8 * len(shuffled_words))
l2 = l1 + int(0.1 * len(shuffled_words))

batch_size = 3

def dataSplit(words):
  context = [0] * batch_size
  X, Y = [], []
  for w in words:
    for ch in list(w + '.'):
      idx = stoi[ch]
      # print('context: ', context, " ch: ", ch, " idx: ", idx)
      X.append(context)
      Y.append(idx)
      context = context[1:]
      context.append(idx)

  return [torch.tensor(X), torch.tensor(Y)]

Xtr, Ytr = dataSplit(words[:l1])
Xde, Yde = dataSplit(words[l1:l2])
Xte, Yte = dataSplit(words[l2:])

print(Xtr.shape, Ytr.shape)
print(Xde.shape, Yde.shape)
print(Xte.shape, Yte.shape)


torch.Size([182778, 3]) torch.Size([182778])
torch.Size([22633, 3]) torch.Size([22633])
torch.Size([22735, 3]) torch.Size([22735])


In [ ]:
# Step 3: Create layers (weights & bias) and embedding tables aka NN (Neural Network)
